## 第14章 元类和抽象基类

### 1.元类

- **元类**：类是元类的实例，就像对象是类的实例一样。更准确地说，每个类都是类型(`type`)的实例，而类型(`type`)就是一个元类。元类允许你重写类的创建方式。
    - 比喻：房屋由蓝图构建，蓝图由模板制作。元类就是那个**模板**。一个模板可以产出多种不同蓝图，每种蓝图可以建造多种不同房屋。
    - 声明：你完全可能在整个职业生涯中**一次都不直接使用元类**，元类是99%的用户都不应该担心的深层魔法。

- 用type()创建类：`类名 = type(类名, (父类1, 父类2, ...), {属性名: 属性值, 方法名: 方法})`。
    - 使用`class`关键字创建类的方式，实际上只是实例化`type`元类的语法糖。`type`元类的好处是可以在运行时**动态创建类**。

In [ ]:
Food = type('Food', (), {})
# 类似→ class Food: pass

def __init__(obj, toppings):
    obj.toppings = toppings

Pizza = type('Pizza', (Food,), {'name':'Pizza', '__init__':__init__})
# 类似：
# class Pizza(Food):
#     def __init__(self, toppings):
#         self.toppings = toppings

print(Pizza.name)           # 类属性
pizza = Pizza('Cheese')
print(pizza.toppings)       # 实例属性

In [ ]:
# 自定义元类
class Gadget(type):
    # 调用type()创建类时调用
    def __new__(self, name, bases, namespace):
        print(f'Creating a {name} gadget')
        return super().__new__(self, name, bases, namespace)

    # 创建一个字典存储正在创建的类的所有方法和类属性
    @classmethod
    def __prepare__(cls, name, bases):
        return {'color': 'red'}

# 自定义元类创建的类，通过metaclass参数指定元类
class Thing(metaclass=Gadget):
    def __init__(self, widget):
        self.widget = widget

    def frob(self):
        print(f'Frobbing {self.widget}')

thing = Thing('Button')     # Creating a Thing gadget
thing.frob()                # Frobbing Button
print(Thing.color)          # red
print(thing.__class__)      # <class '__main__.Thing'>

### 2.抽象基类

- **抽象基类(Abstract Base Class, ABC)**：允许你指定任何继承自该ABC的类**必须实现**的特定接口。如果派生类没有提供预期接口，类实例化将失败。
    - ABC**不能直接实例化**，只能被其他类继承。
    - 通常ABC只定义需要哪些方法，将实际实现留给派生类，不包含任何属性。
    - 在某些情况下，ABC可能提供部分方法的实现。
    - 可以将ABC视为一种**接口契约**：类同意实现ABC指定的方法。

- **内置抽象基类**：Python为迭代器等常见接口提供了内置ABC。
    - `collections.abc`：与容器/迭代相关的抽象类。
    - `numbers`：与数值类型相关的抽象类。
    - `contextlib`：与`with`语句相关的抽象类。
    - `selectors`：与选择器相关的抽象类。
    - `asyncio`：异步相关的抽象类。


In [ ]:
# 使用内置抽象基类
from collections.abc import Sequence
from numbers import Complex

def product_of_thirds(sequence):
    if not isinstance(sequence, Sequence):  # 检查是否为抽象类Sequence的实例
        raise TypeError("Argument must be a sequence")
    if not isinstance(sequence[0], Complex):  # 检查是否为抽象类Complex的实例
        raise TypeError("Sequence elements must support multiplication")

    r = 1
    for x in sequence[2::3]:
        r *= x
    return r

print(product_of_thirds(range(1, 10)))  # 162
print(product_of_thirds('hello'))       # TypeError: Sequence elements must support multiplication

In [ ]:
# 从抽象基类派生
from collections.abc import Collection, Iterator

# 从抽象基类Collection派生，必须实现__iter__、__len__、__contains__等方法
class CafeQueue(Collection):
    def __init__(self):
        self._queue = []
        self._orders = {}
        self._togo = {}

    def __iter__(self):
        return CafeQueueIterator(self)

    def __len__(self):
        return len(self._queue)

    def __contains__(self, customer):
        return (customer in self._queue)

    def add_customer(self, customer, *orders, to_go=True):
        self._queue.append(customer)
        self._orders[customer] = tuple(orders)
        self._togo[customer] = to_go

# 从抽象基类Iterator派生，必须实现__next__、__iter__等方法
class CafeQueueIterator(Iterator):
    def __init__(self, iterable):
        self._iterable = iterable
        self._position = 0

    def __next__(self):
        if self._position >= len(self._iterable):
            raise StopIteration
        customer = self._iterable._queue[self._position]
        orders = self._iterable._orders[customer]
        togo = self._iterable._togo[customer]
        self._position += 1
        return (customer, orders, togo)

    def __iter__(self):
        return self

def serve_customers(queue):
    if not isinstance(queue, Collection):  # 检查queue是否是抽象基类Collection的实例
        raise TypeError("serve_customers() requires a collection.")

    if not len(queue):
        print("Queue is empty.")
        return

    def brew(order):
        print(f"(Making {order}...)")

    for customer, orders, to_go in queue:
        for order in orders: brew(order)
        if to_go:
            print(f"Order for {customer}!")
        else:
            print(f"(Takes order to {customer})")

queue = CafeQueue()
queue.add_customer('Raquel', 'double macchiato', to_go=False)
queue.add_customer('Naomi', 'large mocha, skim')
queue.add_customer('Anmol', 'mango lassi')
serve_customers(queue)

- **自定义抽象基类**：让一个类成为抽象基类的方式如下：
    - 让它继承 `abc.ABC` 或另一个 ABC 类。
    - 给它至少定义一个用 **`@abstractmethod`** 装饰器标记的方法。注意：`@abstractmethod` 必须是**最内层**的装饰器。

In [ ]:
from collections.abc import Collection, Iterator
from abc import abstractmethod

# 自定义抽象基类：从Collection派生，并声明了add_customer方法和first属性
class CustomerQueue(Collection):
    @abstractmethod
    def add_customer(self, customer):pass

    @property
    @abstractmethod
    def first(self):pass

# 继承自CustomerQueue自定义抽象基类
class CafeQueue(CustomerQueue):
    def __init__(self):
        self._queue = []
        self._orders = {}
        self._togo = {}

    def __iter__(self):
        return CafeQueueIterator(self)

    def __len__(self):
        return len(self._queue)

    def __contains__(self, customer):
        return (customer in self._queue)

    def add_customer(self, customer, *orders, to_go=True):
        self._queue.append(customer)
        self._orders[customer] = tuple(orders)
        self._togo[customer] = to_go

    @property
    def first(self):
        return self._queue[0]

class CafeQueueIterator(Iterator):
    def __init__(self, iterable):
        self._iterable = iterable
        self._position = 0

    def __next__(self):
        if self._position >= len(self._iterable):
            raise StopIteration
        customer = self._iterable._queue[self._position]
        orders = self._iterable._orders[customer]
        togo = self._iterable._togo[customer]
        self._position += 1
        return (customer, orders, togo)

    def __iter__(self):
        return self

def serve_customers(queue):
    if not isinstance(queue, CustomerQueue):  # 检查queue是否是CustomerQueue的实例
        raise TypeError("serve_customers() requires a customer queue.")

    if not len(queue):
        print("Queue is empty.")
        return

    def brew(order):
        print(f"(Making {order}...)")

    for customer, orders, to_go in queue:
        for order in orders: brew(order)
        if to_go:
            print(f"Order for {customer}!")
        else:
            print(f"(Takes order to {customer})")

queue = CafeQueue()
queue.add_customer('Raquel', 'double macchiato', to_go=False)
queue.add_customer('Naomi', 'large mocha, skim')
queue.add_customer('Anmol', 'mango lassi')
print(f"The first person in line is {queue.first}.")
serve_customers(queue)

- **虚拟子类**：虚拟子类允许你让抽象基类报告某些类是其派生的，这使得你可以将特定的内置类和第三方类指定为实现了自定义抽象基类所需的接口。
    - `register()`：让某个抽象基类**声称**另外一个类是其派生类之一，而不需要这个类实际继承自该抽象基类。缺陷：只能一个个类注册，如果有多个类需要注册时比较麻烦。
    - `__subclasshook__()`方法：当一个类尝试继承自抽象基类时，该方法会被调用，以确定是否允许继承。

In [ ]:
from abc import ABC, abstractmethod
from collections.abc import Sequence

# 定义一个抽象基类，用于表示回文字符串
class Palindromable(ABC):

    @abstractmethod
    def __str__(self): pass

    @abstractmethod
    def __iter__(self): pass

    @abstractmethod
    def __reversed__(self): pass

    @classmethod
    def __subclasshook__(cls, C):  # 定义一个子类钩子方法，用于检查是否为Sequence的子类
        if issubclass(C, Sequence):
            return True
        return NotImplemented

# Palindromable.register(Sequence)  # 将Sequence类注册为Palindromable的虚拟子类

# 定义一个具体实现类，用于表示字母回文字符串
class LetterPalindrome(Palindromable):
    def __init__(self, string):
        self._raw = string
        self._stripped = ''.join(filter(str.isalpha, string.lower()))

    def __str__(self):
        return self._raw

    def __iter__(self):
        return self._stripped.__iter__()

    def __reversed__(self):
        return reversed(self._stripped)

# 检查一个序列是否为回文字符串
def check_palindrome(sequence):
    if not isinstance(sequence, Palindromable):
        raise TypeError("Cannot check for palindrome on that type.")

    for c, r in zip(sequence, reversed(sequence)):
        if c != r:
            return False
    return True

canal = LetterPalindrome("A man, a plan, a canal - Panama!")
bolton = LetterPalindrome("Bolton")
print(check_palindrome(canal))                          # True
print(check_palindrome(bolton))                         # False

print(check_palindrome([1, 2, 3, 2, 1]))                # prints 'True'
print(check_palindrome((1, 2, 3, 2, 1)))                # prints 'True'
print(check_palindrome(LetterPalindrome('race car')))   # prints 'True'
print(check_palindrome({1, 2, 3, 2, 1}))                # TypeError: Cannot check for palindrome on that type.

### 3.本章小结

```text
元类与抽象基类
│
├── 一、元类（Metaclasses）⭐⭐
│   ├── 核心概念：类是 type 的实例，type 是元类
│   ├── type() 创建类（动态，但很少需要）
│   ├── 自定义元类：继承 type，重写 __new__()
│   │   ├── __new__(self, name, bases, namespace) → 控制类创建
│   │   ├── __prepare__(cls, name, bases) → 返回初始命名空间字典
│   │   └── metaclass=XXX 语法指定元类
│   ├── GOTCHA：多重继承 + 元类冲突 ⚠️
│   ├── ⚠️ 99%的用户不需要元类（Tim Peters）
│   └── __init_subclass__() 钩子 ⭐ → 轻量替代
│       ├── 不需要自定义元类
│       ├── 感知子类创建，参数化子类定义
│       └── Bulletin 示例：Task/Event/Note
│
├── 二、鸭子类型下的类型期望
│   ├── EAFP：捕获异常 → 适合自提供参数的场景
│   ├── LBYL：检查属性（hasattr）
│   │   ├── 陷阱1：可迭代 ≠ 可下标
│   │   ├── 陷阱2：属性存在 ≠ 行为正确
│   │   ├── 陷阱3：方法存在 ≠ 语义正确（如 str.__mul__）
│   │   └── 陷阱4：继承导致误判（如 object.__ge__）
│   └── LBYL：检查接口（ABC）⭐ → 本章核心
│
├── 三、抽象基类（ABC）定义
│   ├── 不能直接实例化，只能被继承
│   ├── 接口契约：派生类必须实现指定方法
│   └── ⚠️ 不等同于 C++/Java 的虚函数
│
├── 四、内置 ABC
│   ├── collections.abc（Container, Sized, Iterable, Iterator, 
│   │   Collection, Sequence...）
│   ├── numbers（Complex, Real, Rational, Integral...）
│   └── isinstance() + ABC → 检查完整行为契约
│
├── 五、从 ABC 派生
│   ├── CafeQueue 示例：继承 Collection + Iterator
│   ├── 优势1：可被标准库 ABC 检查功能
│   └── 优势2：保险策略，防止关键方法被意外删除
│
├── 六、实现自定义 ABC ⭐
│   ├── 继承 abc.ABC + @abstractmethod
│   ├── @abstractmethod 必须是最内层装饰器
│   ├── 抽象方法可有默认实现（通过 super() 调用）
│   └── CustomerQueue 示例
│
├── 七、Palindromable ABC 完整示例 ⭐
│   ├── 定义 ABC（@abstractmethod × 3）
│   ├── LetterPalindrome 派生类
│   └── check_palindrome() 函数（isinstance + ABC）
│
└── 八、虚拟子类化 ⭐⭐
    ├── register() 方法 → 注册单个类
    ├── register(Sequence) → 注册整个 ABC 层级
    └── __subclasshook__() ⭐ → 自动化虚拟子类检查
        ├── 返回 True/False/NotImplemented
        └── ⚠️ 必须返回 NotImplemented（非 False）
```